In [ ]:
import kagglehub
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")

%pip install kagglehub catboost xgboost tqdm -q


print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
df = pd.read_csv(f"{path}/Q3_data.csv")

In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 1: Write your code here:
missing = df.isnull().sum() / len(df) * 100

missing_data = pd.DataFrame({"Columns": missing.index, 'Missing_Percentage': missing.values})

missing_data = missing_data[missing_data['Missing_Percentage'] > 0].sort_values('Missing_Percentage', ascending=False)

df.fillna(0, inplace = True)

In [ ]:
# Task 2: Write your code here:
print(df.duplicated().sum())
#No duplicates

In [ ]:
# Task 3: Write your code here:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
categories = df.select_dtypes(include=["object"]).columns

for col in categories:
  df[col] = le.fit_transform(df[col])

df.head()

In [ ]:
# Task 4: Write your code here:
from sklearn.preprocessing import StandardScaler
feature = df.select_dtypes(include=["float64"]).columns
standard_scaler = StandardScaler()
df[feature] = standard_scaler.fit_transform(df[feature])

df.head()

In [ ]:
# Task 5: Write your code here:
def check_target_imbalance(df, target_column):
  print("Target Distribution:")
  print(df[target_column].value_counts(normalize=True))
  sns.countplot(x=df[target_column])
  plt.title("Target Distribution")
  plt.show()

check_target_imbalance(df, "Target")

In [ ]:
# Task 1: Write your code here:
X = df.drop('Target', inplace = False, axis = 1)
y = df['Target']

In [ ]:
# Task 2,3,4,5: Write your code here:
from sklearn.model_selection import StratifiedKFold
from catboost import CatBoostClassifier
from sklearn.metrics import f1_score, accuracy_score

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
model = CatBoostClassifier(
      verbose=0,
      n_estimators=320,
      max_depth=4
  )

acc_fold = []
f1_fold = []

for fold_idx, (train_index, test_index) in enumerate(skf.split(X, y)):

  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  model.fit(X_train, y_train)
  y_pred = model.predict(X_test)

  accuracy = accuracy_score(y_test, y_pred)
  f1 = f1_score(y_test, y_pred, zero_division=0)

  acc_fold.append(accuracy)
  f1_fold.append(f1)

  print(f"fold no. {fold_idx}, acc = {accuracy}, f1 = {f1}")


In [ ]:
# Task 1: Write your code here:
feature = df.drop("Target", inplace = False, axis = 1).columns
feature_importance = pd.DataFrame({
    'feature': feature,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Task 2: Write your code here:
print(feature_importance.iloc[0])

In [ ]:
# Task Bonus: Write your code here:
X1 = df[feature_importance.iloc[0]['feature']]

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
model = CatBoostClassifier(
      verbose=0,
      n_estimators=320,
      max_depth=4
  )

acc1_fold = []
f11_fold = []

for fold_idx, (train_index, test_index) in enumerate(skf.split(X1, y)):

  # 1. Split data
  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  model.fit(X_train, y_train) # train
  y_pred = model.predict(X_test) # validate

  # 3. Save metrics for that model in this fold
  accuracy = accuracy_score(y_test, y_pred)
  f1 = f1_score(y_test, y_pred, zero_division=0)

  acc1_fold.append(accuracy)
  f11_fold.append(f1)

  print(f"fold no. {fold_idx}, acc = {accuracy}, f1 = {f1}")

In [ ]:
print("This is what I got for our original dataframe : ", acc_fold)
print("This is what I got for with the golden feature : ", acc1_fold)